# openEO Basics Exercise

Welcome to the openEO basics exercise!

## Learning Objectives

By the end of this exercise, you will be able to:

- Understand the core concepts of openEO: Processes, Collections, and Jobs
- Connect to an openEO backend and browse available data
- Build processing workflows using the openEO process graph
- Filter data using spatial and temporal parameters
- Apply basic processing operations like NDVI calculation
- Execute and download results from openEO
- Complete practical exercises to reinforce your learning

**Let's get started!**

## 1. Install Required Libraries

Before we begin, we need to install the necessary Python libraries for working with openEO data. Run the cell below to install all required packages.

In [ ]:
!pip install openeo

## 2. Understanding openEO Components

openEO is an open API for cloud-based processing of large EO datasets. It provides a common interface to access and process satellite data from different cloud platforms without having to learn platform-specific APIs.

openEO organizes EO processing in a structured way with three main components:

### Collections
- EO datasets (e.g., Sentinel-2, Landsat)
- Contain metadata about spatial/temporal extent, bands, etc.
- Available from different data providers

### Processes
- Building blocks for data processing workflows
- Include operations like filtering, mathematical operations, aggregations
- Can be chained together to create complex workflows

### Jobs
- Executable processing workflows
- Can be synchronous (immediate) or batch (queued)
- Return processed results that can be downloaded

Let's see this in action by exploring these concepts!

## 3. Connecting to an openEO Backend

Now let's connect to a real openEO backend! We'll use the **Copernicus Data Space Ecosystem (CDSE)** platform.

In [ ]:
# Import openEO library
import openeo as eo

In [ ]:
# Connect to the openEO Platform
backend_url = "https://openeo.dataspace.copernicus.eu"  # openEO backend URL address of CDSE
connection = eo.connect(backend_url).authenticate_oidc()

print("Connected to openEO Platform!")
print(f"  Backend URL: {backend_url}")
print(f"  API Version: {connection.version_info()['api']}")

## 4. Exploring Available Collections

Let's explore what Earth observation collections are available on this backend.

In [ ]:
# List available collections
collections = connection.list_collections()

print(f"Total collections available: {len(collections)}")
print("Collections:")
for i, collection in enumerate(collections):
    print(f"  {i+1}. {collection['id']} - {collection.get('title', 'No title')}")

## 5. Examining a Specific Collection

Let's examine the Sentinel-2 L2A collection in detail to understand its properties.

In [ ]:
# Get Sentinel-2 L2A collection details
s2_collection = connection.describe_collection("SENTINEL2_L2A")

print("Sentinel-2 L2A collection details:")
print(f"  Title: {s2_collection.get('title', 'N/A')}")
print(f"  Description: {s2_collection.get('description', 'N/A')}")
print(f"  Temporal Extent: {s2_collection.get('extent', {}).get('temporal', 'N/A')}")
print(f"  Spatial Extent: {s2_collection.get('extent', {}).get('spatial', 'N/A')}")

# Show available bands
bands = s2_collection.get('summaries', {}).get('eo:bands', [])
print(f"  Available Bands ({len(bands)} total):")
for band in bands:
    print(f"  - {band.get('name', 'N/A')}: {band.get('common_name', 'N/A')}")

## 6. Building a Processing Workflow

Now let's build our first openEO processing workflow! We'll load Sentinel-2 data over Amsterdam, Netherlands and apply some basic filtering.

In [ ]:
# Define spatial and temporal extent for Amsterdam
spatial_extent = {
    "west": 4.7,
    "south": 52.3,
    "east": 5.0,
    "north": 52.4
}

temporal_extent = ["2025-06-01", "2025-06-30"]  # June 2025

# Load the collection
datacube = connection.load_collection(
    collection_id="SENTINEL2_L2A",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=["B04", "B03", "B02", "B08"]  # Red, Green, Blue, NIR
)

print("Processing workflow created.")

## 7. Calculate NDVI

Let's add an NDVI calculation to our workflow. NDVI values near 1 signify healthy vegetation, while values near 0 indicate bare soil or water.

In [ ]:
# Calculate NDVI
red = datacube.band("B04") 
nir = datacube.band("B08") 
ndvi = (nir - red)/(nir + red)

# Reduce temporal dimension by taking the mean
ndvi_mean = ndvi.reduce_dimension(
    dimension="t",
    reducer="mean"
)

print("NDVI calculation added to workflow.")

## 8. Execute the Workflow

Now let's execute our workflow synchronously to get a preview of the results.

In [ ]:
try:
    ndvi_mean.download("ndvi_mean.tiff")  # Execute synchronously
    
    print("Workflow executed successfully. Check saved TIFF file.")

except Exception as e:
    print(f"Execution failed: {e}")
    print("This might be due to authentication requirements or backend limitations.")

## 9. Batch Job to Execute the Workflow

Create a batch job to save the NDVI results. This will be a "dry run" to understand the process.

In [ ]:
# Define output format
res = ndvi_mean.save_result(format="GTiff")

# Create a batch job (without actually submitting)
job = connection.create_job(
    res,
    title="NDVI Analysis",
    description="NDVI calculation using Sentinel-2 data"
)
    
print("Batch job created successfully.")

## 10. Visualize Process Graph

openEO workflows are represented as process graphs, which show how openEO processes are chained together. Each node represents a processing step.

Let's visualize what our NDVI workflow looks like.

In [ ]:
# Display the process graph
ndvi_mean

Process graph can be represented in JSON.

In [ ]:
# Get the process graph as JSON
process_graph = ndvi_mean.to_json()

# Display the JSON representation
print(process_graph)

You can also display JSON representation interactively.

In [ ]:
# Import libraries
import json
from IPython.display import JSON

In [ ]:
# Display the JSON tree
JSON(json.loads(process_graph))

## 11. Run the Batch Job

Let's run the batch job and get the results.

In [ ]:
# Start the job and wait until it finished
job.start_and_wait()

# Download the results
job.get_results().download_files("output")

# Conclusion

Congratulations, you have successfully completed the openEO basics exercise!

## What You've Learned

- **Backend Connection**: Connecting to openEO platforms and exploring available data
- **Data Discovery**: Understanding collections, their properties, and available bands
- **Workflow Building**: Creating processing workflows with spatial and temporal filtering
- **Data Processing**: Applying mathematical operations like NDVI calculation
- **Process Graphs**: Understanding how openEO represents computational workflows
- **Job Management**: Understanding how to create and submit batch processing jobs

## Next Steps

Now that you understand openEO basics, you can:

1. **Explore More Collections**: Try Landsat, Sentinel-1 SAR, or other datasets
2. **Advanced Processing**: Learn about machine learning processes, aggregations, and custom functions
3. **Large Scale Analysis**: Process data over larger areas and longer time periods
4. **Integration**: Combine openEO with other tools like STAC for data discovery

## Additional Resources

- [openEO Documentation](https://openeo.org/documentation/)
- [openEO Python Client](https://openeo.readthedocs.io/)
- [openEO Platform](https://openeo.dataspace.copernicus.eu/)
- [openEO Processes](https://processes.openeo.org/)

**Happy EO data processing with openEO!**